In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
from tqdm import tqdm

Load census variables and parse for relevant information

In [16]:
df_cen_cma_data = pd.read_csv('../data/census/98-401-X2021002_eng_CSV/98-401-X2021002_English_CSV_data.csv', encoding='latin')
df_cen_cma_data = df_cen_cma_data[['DGUID', 'GEO_LEVEL', 'GEO_NAME', 'CHARACTERISTIC_ID', 'CHARACTERISTIC_NAME', 'C1_COUNT_TOTAL', 'C10_RATE_TOTAL']]

# NOTE: later we might want to add more characteristics
df_cen_cma_data = df_cen_cma_data[df_cen_cma_data['CHARACTERISTIC_ID'] == 1].reset_index(drop=True)

In [20]:
df_ada_cma_rel = pd.read_csv('../data/census/ada_cma_relation.csv')

In [23]:
df_ada_cma_rel

,ADADGUID_ADAIDUGD,CMADGUID_RMRIDUGD
0,2021S051610010007,2021S0503001
1,2021S051610010015,2021S0503001
2,2021S051610010018,2021S0503001
3,2021S051610010016,2021S0503001
4,2021S051610010013,2021S0503001
...,...,...
5428,2021S051662080008,NaN
5429,2021S051662080007,NaN
5430,2021S051662080002,NaN
5431,2021S051662080004,NaN


In [ ]:
# gdf_cma = gpd.read_file('../data/census/lcma000b21a_e')
# gdf_cma = gdf_cma[['CMAUID', 'DGUID', 'CMANAME', 'PRUID', 'geometry']]

Load tariff information and join to CMAs

In [21]:
df_tariffs_ada = pd.read_excel('tariff-impacts-ada-data.xlsx', sheet_name='Counts')

ADADGUID  Auto_B  Alum_B  Steel_B  Cop_B  Lum_B  Ene_B  \
0     2021S051610010001       0       1        1      0      2      0   
1     2021S051610010002       0       0        0      0      0      0   
2     2021S051610010003       0       1        1      0      4      0   
3     2021S051610010004       0       1        1      1      0      2   
4     2021S051610010005       2       1        1      0      0      0   
...                 ...     ...     ...      ...    ...    ...    ...   
5428  2021S051662080004       0       0        0      0      0      0   
5429  2021S051662080005       0       0        0      0      0      0   
5430  2021S051662080006       0       0        0      0      0      0   
5431  2021S051662080007       0       0        0      0      0      0   
5432  2021S051662080008       0       0        0      0      0      0   

      CUSMA_B  Total_B  Auto_E  ...  Total_E  Auto_C  Alum_C  Steel_C  Cop_C  \
0          16       16       0  ...      844       0       1        1      0   
1           2        2       0  ...       10       6       8        8      1   
2           9        9       0  ...       54       0       1        2      0   
3           7        7       0  ...       55       6       9        8      1   
4           5        5       5  ...       29       6       9        8      1   
...       ...      ...     ...  ...      ...     ...     ...      ...    ...   
5428        0        0       0  ...        0       0       0        0      0   
5429        0        0       0  ...        0       0       0        0      0   
5430        0        0       0  ...        0       0       0        0      0   
5431        0        0       0  ...        0       0       0        0      0   
5432        0        0       0  ...        0       0       0        0      0   

      Lum_C  Ene_C  CUSMA_C  Total_C  \
0         5      0      613      613   
1         3      7       82       82   
2         8     28      258      258   
3         4     13       95       95   
4         4     15      107      107   
...     ...    ...      ...      ...   
5428      0      0        0        0   
5429      0      0        0        0   
5430      0      0        0        0   
5431      0      0        0        0   
5432      0      0        0        0   

                                           geometry  
0      POINT (-53.25418982995286 47.86225405767045)  
1      POINT (-52.76053689500512 47.72308896969448)  
2      POINT (-53.26648981737229 47.73592881595869)  
3       POINT (-52.71311539417454 47.6217728223513)  
4     POINT (-52.77029796102484 47.647250554620264)  
...                                             ...  
5428  POINT (-115.37130429560911 67.80896543507862)  
5429   POINT (-95.88321607603997 68.64144026686964)  
5430   POINT (-89.80822157846997 68.53257590504529)  
5431  POINT (-107.82111393211768 67.68532480911229)  
5432  POINT (-108.11144919560697 66.83117762701185)  

[5433 rows x 26 columns]

In [ ]:
# Prepare the relation dataframe
df_ada_cma_rel_clean = df_ada_cma_rel[['CMADGUID_RMRIDUGD']].rename(columns={'CMADGUID_RMRIDUGD': 'CMADGUID'})
df_ada_cma_rel_clean['ADADGUID'] = df_ada_cma_rel['ADADGUID_ADAIDUGD']

# Prepare the tariffs dataframe (drop geometry column)
df_tariffs_ada_clean = df_tariffs_ada.drop(columns=['geometry'])

# Join ADA tariffs with CMA relation
df_tariffs_joined = df_tariffs_ada_clean.merge(df_ada_cma_rel_clean, on='ADADGUID', how='left')

# Filter out ADAs that are not in any CMA (where CMADGUID is NaN)
df_tariffs_cma_filtered = df_tariffs_joined.dropna(subset=['CMADGUID'])

# Group by CMADGUID and sum all tariff columns
tariff_columns = [col for col in df_tariffs_cma_filtered.columns if col not in ['ADADGUID', 'CMADGUID']]
df_tariffs_cma = df_tariffs_cma_filtered.groupby('CMADGUID')[tariff_columns].sum().reset_index()

print(f"Shape of CMA tariffs dataframe: {df_tariffs_cma.shape}")
df_tariffs_cma.head()